# ClaimGuard production-aligned SLM benchmark
Team Claimix. Maram Kouki contributed the initial B2 model, quantization and adversarial study.

Six candidates, real engine findings, production prompts/verifier and reviewer assistant graph. Synthetic data only. No model has won this protocol yet. Select a CUDA GPU runtime. T4 has no native BF16: use a named FP16 reference against NF4. Unsupported/OOM configurations stay visible. See `docs/26-Production-SLM-Benchmark.md` for the full protocol and human review rubric.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

REPO_REF = "stream/b2/slm-benchmark-v2"  # Use a full commit SHA for a published experiment.
REPO = Path("/content/ClaimGuard")
if not REPO.exists():
    subprocess.run(  # noqa: S603 - fixed experiment command, no shell
        ["/usr/bin/git", "clone", "https://github.com/OussemaHarrabi/ClaimGuard.git", str(REPO)],
        check=True,
    )
    subprocess.run(["/usr/bin/git", "checkout", REPO_REF], cwd=REPO, check=True)  # noqa: S603 - trusted notebook configuration, no shell
os.chdir(REPO)
print(
    "Frozen source:",
    subprocess.check_output(["/usr/bin/git", "rev-parse", "HEAD"], text=True).strip(),
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "notebooks/requirements-slm.txt"], check=True
)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev,agent]"], check=True)

## Credentials and runtime
Accept each model's current license where required. Optional Colab secret: `HF_TOKEN`. Never paste tokens into cells or commit/export them. Restart if packages were previously imported. Persist outputs on a mounted Drive to survive Colab resets. Model downloads and runs may take hours.

In [ ]:
import torch
import transformers
from huggingface_hub import login

try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False)
    del token
except Exception as exc:  # noqa: BLE001 - optional Colab secret, never echo token
    print("No HF secret loaded; public models can be attempted:", type(exc).__name__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("Native BF16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
assert torch.cuda.is_available(), "Select a CUDA GPU runtime."
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/benchmark",
        "tests/edu_intake/test_phase1_samples.py",
        "-q",
    ],
    check=True,
)

## 1. Screening
The default 60-case smoke test is **ineligible for selection**. Set `LIMIT=0` for 1,080 screening findings. Keep the protocol fixed before opening release cases. `--followups` exercises corrections, evidence questions, history, injection, payer decisions, foreign-clinic requests, clinical advice and off-topic questions through the real guard/verify/one-repair/fallback graph. Refused questions must make no model calls.

In [ ]:
MODELS = ["gemma4-e4b", "phi4-mini", "qwen3-4b", "smollm3-3b", "lfm25-12b", "qwen35-4b"]
PRECISIONS = ["nf4"]  # Add fp16; use bf16 only on a native-capable GPU.
LIMIT = 60
SCREEN_DIR = Path("/content/claimix-benchmark/screen-01")
subprocess.run(  # noqa: S603 - fixed experiment command, no shell
    [
        sys.executable,
        "-m",
        "notebooks.slm_benchmark_runner",
        "run",
        "--output",
        str(SCREEN_DIR),
        "--split",
        "screen",
        "--variants",
        "3",
        "--limit",
        str(LIMIT),
        "--followups",
        "--models",
        *MODELS,
        "--precisions",
        *PRECISIONS,
    ],
    check=True,
)

## 2. Independent human review
Inspect raw drafts, served answers, actual finding/evidence/rule, rejection reasons and timings. Fill a copy named `semantic-review-filled.csv` of the exported template. Two different reviewers per exact output: `unsupported` and `correction_safe` must be `true`/`false`; clarity/usefulness are 1–5. Discuss disagreements, then provide a fresh agreed independent pair. Missing/disputed labels stay unknown. Review follow-up answers separately against their question/history. Do not infer semantic correctness from citations.

In [ ]:
import csv
import json

FILLED = SCREEN_DIR / "semantic-review-filled.csv"
LABELS = SCREEN_DIR / "semantic-review.json"
if FILLED.exists():
    labels = []
    for row in csv.DictReader(FILLED.open(encoding="utf-8")):
        if not all(
            row.get(f, "").strip()
            for f in ["unsupported", "correction_safe", "clarity", "usefulness"]
        ):
            continue
        for field in ["unsupported", "correction_safe"]:
            value = row[field].strip().lower()
            assert value in ["true", "false"], f"Use true/false for {field}"
            row[field] = value == "true"
        for field in ["clarity", "usefulness"]:
            row[field] = int(row[field])
        labels.append(row)
    LABELS.write_text(json.dumps(labels, ensure_ascii=False, indent=2), encoding="utf-8")
    subprocess.run(  # noqa: S603 - fixed experiment command, no shell
        [
            sys.executable,
            "-m",
            "notebooks.slm_benchmark_runner",
            "report",
            "--output",
            str(SCREEN_DIR),
            "--models",
            *MODELS,
            "--precisions",
            *PRECISIONS,
            "--reviews",
            str(LABELS),
        ],
        check=True,
    )
else:
    print("No completed reviews. Semantic quality remains UNKNOWN.")

## 3. Frozen held-out confirmation
Choose finalists from screening evidence, not the illustrative placeholders below. Flip `RUN_RELEASE` only after freezing prompts/models/decoding. Three variants produce 810 held-out findings across 18 families. These are correlated synthetic examples, **not 810 independent clinical observations**. Same model revision/cases/prompt/generation settings across precisions. Missing reference or incomplete reviews means quantization quality is not established. Check per-rule/status strata and follow-up human semantics, not just aggregate scores.

In [ ]:
RUN_RELEASE = False
FINALISTS = ["gemma4-e4b", "smollm3-3b"]  # Placeholders, NOT a ranking.
RELEASE_DIR = Path("/content/claimix-benchmark/release-01")
if RUN_RELEASE:
    subprocess.run(  # noqa: S603 - fixed experiment command, no shell
        [
            sys.executable,
            "-m",
            "notebooks.slm_benchmark_runner",
            "run",
            "--output",
            str(RELEASE_DIR),
            "--split",
            "release",
            "--variants",
            "3",
            "--followups",
            "--models",
            *FINALISTS,
            "--precisions",
            "fp16",
            "nf4",
        ],
        check=True,
    )
else:
    print("Held-out evaluation locked until screening review is complete.")

## 4. Export
Return manifests, revisions, deterministic baseline, raw/served outputs, rejection/repair/fallback/refusal traces, latency/VRAM and completed reviews. After release, rerun the labeling/report cell with `SCREEN_DIR=RELEASE_DIR`, `MODELS=FINALISTS`, and the release precisions. There is no auto-deployment or zero-hallucination claim. Fine-tuning is a later controlled challenger experiment using only development families.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/claimix-slm-results", "zip", SCREEN_DIR)
files.download(archive)